# 基于相图分析的镍基高温合金微观组织预测

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils import shuffle

/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/macbookpro/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
features = pd.read_excel('镍基高温合金微观数据集合的副本.xlsx')
features

,Ni,Cr,Co,Fe,Al,Ti,Nb,Mo,W,C,B,Zr,强化相,液态温度,固态温度,凝固区间温度,强化相溶解温度,加工窗口
0,53.3950,17.74,0.31,18.579,0.53,0.97,5.48,2.97,0.00,0.026,0.0000,0.000,8.20,1360.00,1100.0,260.00,899.35,200.65
1,52.4000,19.01,0.02,18.837,0.57,1.00,5.07,3.06,0.00,0.030,0.0030,0.000,8.97,1358.19,1090.0,268.19,916.20,173.80
2,55.4315,18.93,11.00,0.000,1.57,3.13,0.00,9.83,0.00,0.075,0.0035,0.030,27.23,1331.25,1175.0,156.25,1056.06,118.94
3,53.2700,18.60,0.00,17.746,1.20,0.80,5.30,3.00,0.00,0.080,0.0040,0.000,13.59,1358.32,1095.0,263.32,938.05,156.95
4,51.6890,19.20,0.00,19.301,0.70,1.01,5.05,3.05,0.00,0.000,0.0000,0.000,10.48,1361.27,1090.0,271.27,932.34,157.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,56.7700,15.70,12.90,0.000,2.10,3.70,0.70,4.00,4.00,0.050,0.0300,0.050,37.66,1349.01,1145.0,204.01,1101.72,43.28
120,56.8500,15.84,12.59,0.093,2.09,3.73,0.80,3.94,3.98,0.038,0.0120,0.037,38.02,1349.32,1140.0,209.32,1106.25,33.75
121,74.0931,20.45,0.00,0.000,0.98,2.87,1.57,0.00,0.00,0.030,0.0069,0.000,17.81,1364.31,1135.0,229.31,948.97,186.03
122,62.6180,12.98,8.00,0.000,3.48,2.55,3.50,3.40,3.40,0.060,0.0120,0.000,51.88,1340.72,1095.0,245.72,1148.15,53.15


In [4]:
features = pd.read_excel('镍基高温合金微观数据集合的副本.xlsx')
label2 = np.array(features['强化相'])
label3 = np.array(features['液态温度'])
label4 = np.array(features['固态温度'])
label5 = np.array(features['凝固区间温度'])
label6 = np.array(features['强化相溶解温度'])
label7 = np.array(features['加工窗口'])
features= features.drop('强化相', axis = 1)
features= features.drop('液态温度', axis = 1)
features= features.drop('固态温度', axis = 1)
features= features.drop('凝固区间温度', axis = 1)
features= features.drop('强化相溶解温度', axis = 1)
features= features.drop('加工窗口', axis = 1)

In [5]:
features

,Ni,Cr,Co,Fe,Al,Ti,Nb,Mo,W,C,B,Zr
0,53.3950,17.74,0.31,18.579,0.53,0.97,5.48,2.97,0.00,0.026,0.0000,0.000
1,52.4000,19.01,0.02,18.837,0.57,1.00,5.07,3.06,0.00,0.030,0.0030,0.000
2,55.4315,18.93,11.00,0.000,1.57,3.13,0.00,9.83,0.00,0.075,0.0035,0.030
3,53.2700,18.60,0.00,17.746,1.20,0.80,5.30,3.00,0.00,0.080,0.0040,0.000
4,51.6890,19.20,0.00,19.301,0.70,1.01,5.05,3.05,0.00,0.000,0.0000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...
119,56.7700,15.70,12.90,0.000,2.10,3.70,0.70,4.00,4.00,0.050,0.0300,0.050
120,56.8500,15.84,12.59,0.093,2.09,3.73,0.80,3.94,3.98,0.038,0.0120,0.037
121,74.0931,20.45,0.00,0.000,0.98,2.87,1.57,0.00,0.00,0.030,0.0069,0.000
122,62.6180,12.98,8.00,0.000,3.48,2.55,3.50,3.40,3.40,0.060,0.0120,0.000


In [6]:
n_samples, n_features = features.shape
features = pd.get_dummies(features)
feature_list = list(features.columns)#获取列名
features = np.array(features)#获取影响因子

In [7]:
from sklearn.model_selection import train_test_split
test_ratio = 0.25#按照五比一划分数据集合；4/5的数据用于训练集合，1/5的数据用于验证集合。
SEED = 26 ### the test/train data is checked on this seed, it has similiar distribution to the whole dataset
train_features2, test_features2, train_labels2, test_labels2 = train_test_split(features, label2,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features3, test_features3, train_labels3, test_labels3 = train_test_split(features, label3,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features4, test_features4, train_labels4, test_labels4 = train_test_split(features, label4,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features5, test_features5, train_labels5, test_labels5 = train_test_split(features, label5,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features6, test_features6, train_labels6, test_labels6 = train_test_split(features, label6,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)
train_features7, test_features7, train_labels7, test_labels7 = train_test_split(features, label7,
                                                                            test_size = test_ratio,
                                                                            random_state = SEED)

In [8]:
model_seed = 100 
from sklearn.ensemble import RandomForestRegressor
Model2 = RandomForestRegressor(random_state=model_seed)
Model3 = RandomForestRegressor(random_state=model_seed)
Model4 = RandomForestRegressor(random_state=model_seed)
Model5 = RandomForestRegressor(random_state=model_seed)
Model6 = RandomForestRegressor(random_state=model_seed)
Model7 = RandomForestRegressor(random_state=model_seed)

In [10]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
def evaluate(true_labels, pred_labels):
    errors = abs(pred_labels - true_labels)
    MAE = mean_absolute_error(true_labels, pred_labels)
    mape = 100 * np.mean(errors / true_labels)
    r2 = r2_score(true_labels, pred_labels)
    RMSE = mean_squared_error(true_labels, pred_labels,squared=False)
    return mape, MAE, RMSE, r2

In [11]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import LeaveOneOut
from numpy import absolute
from numpy import mean
from numpy import std
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_squared_error
cv = LeaveOneOut()
Model2.fit(train_features2,train_labels2)
Model3.fit(train_features3,train_labels3)
Model4.fit(train_features4,train_labels4)
Model5.fit(train_features5,train_labels5)
Model6.fit(train_features6,train_labels6)
Model7.fit(train_features7,train_labels7)


from sklearn.model_selection import cross_val_predict
y_pred2 = cross_val_predict(Model2, features, label2, cv=cv)
y_pred3 = cross_val_predict(Model3, features, label3, cv=cv)
y_pred4 = cross_val_predict(Model4, features, label4, cv=cv)
y_pred5 = cross_val_predict(Model5, features, label5, cv=cv)
y_pred6 = cross_val_predict(Model6, features, label6, cv=cv)
y_pred7 = cross_val_predict(Model7, features, label7, cv=cv)

model_accuracy2, model_MAE2, model_RMSE2, model_r22 = evaluate(label2, y_pred2)
model_accuracy3, model_MAE3, model_RMSE3, model_r23 = evaluate(label3, y_pred3)
model_accuracy4, model_MAE4, model_RMSE4, model_r24 = evaluate(label4, y_pred4)
model_accuracy5, model_MAE5, model_RMSE5, model_r25 = evaluate(label5, y_pred5)
model_accuracy6, model_MAE6, model_RMSE6, model_r26 = evaluate(label6, y_pred6)
model_accuracy7, model_MAE7, model_RMSE7, model_r27 = evaluate(label7, y_pred7)


final_output = pd.DataFrame()
final_output.loc[:,2] = [model_accuracy2, model_MAE2, model_RMSE2, model_r22]
final_output.loc[:,3] = [model_accuracy3, model_MAE3, model_RMSE3, model_r23]
final_output.loc[:,4] = [model_accuracy4, model_MAE4, model_RMSE4, model_r24]
final_output.loc[:,5] = [model_accuracy5, model_MAE5, model_RMSE5, model_r25]
final_output.loc[:,6] = [model_accuracy6, model_MAE6, model_RMSE6, model_r26]
final_output.loc[:,7] = [model_accuracy7, model_MAE7, model_RMSE7, model_r27]



final_output=pd.DataFrame(final_output)
final_output=final_output.T
final_output.columns = ['MAPE', 'MAE', 'RMSE', 'R2']
final_output.index = ['强化相体积分数', ' 液相温度','固相温度',
                      '凝固区间','强化相溶解温度','加工窗口']
final_output.to_excel(r'final_output_LOOCV1.xlsx', index = True)

/var/folders/sy/gl301n8s5pl2fptdx8bcns5h0000gn/T/ipykernel_81963/2342038624.py:48: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  final_output.to_excel(r'final_output_LOOCV1.xlsx', index = True)


# 对新数据进行预测

In [12]:
data=pd.read_excel('高温合金数据集的副本2.xlsx')
data

,Ni,Cr,Co,Fe,Al,Ti,Nb,Mo,W,C,B,Zr
0,53.395,17.740,0.310,18.579,0.530,0.970,5.480,2.970,0.000,0.026,0.000,0.000
1,53.395,17.740,0.310,18.579,0.530,0.970,5.480,2.970,0.000,0.026,0.000,0.000
2,53.395,17.740,0.310,18.579,0.530,0.970,5.480,2.970,0.000,0.026,0.000,0.000
3,53.395,17.740,0.310,18.579,0.530,0.970,5.480,2.970,0.000,0.026,0.000,0.000
4,52.400,19.010,0.020,18.837,0.570,1.000,5.070,3.060,0.000,0.030,0.003,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,60.978,18.950,6.652,4.101,2.017,1.237,1.264,4.667,5.960,0.030,0.000,0.000
1016,54.951,19.798,6.549,3.684,1.642,2.047,1.194,4.821,2.808,0.030,0.000,0.000
1017,55.125,16.855,8.123,0.121,2.001,2.959,0.233,6.241,2.239,0.030,0.000,0.000
1018,55.324,16.742,7.462,2.760,1.598,2.774,0.752,5.617,2.724,0.040,0.000,0.025


In [13]:
new_data_x_scaled=np.array(data)#获取影响因子
predicted_y2 = Model2.predict(new_data_x_scaled)
predicted_y3 = Model3.predict(new_data_x_scaled)
predicted_y4 = Model4.predict(new_data_x_scaled)
predicted_y5 = Model5.predict(new_data_x_scaled)
predicted_y6 = Model6.predict(new_data_x_scaled)
predicted_y7 = Model7.predict(new_data_x_scaled)
result_df = pd.DataFrame({'X': new_data_x_scaled.tolist(),
                             '强化相体积分数': predicted_y2,'液相温度': predicted_y3,'固相温度': predicted_y4,
                             '凝固区间': predicted_y5,'强化相溶解温度': predicted_y6,'加工窗口': predicted_y7,
                             })#将预测的真实值进行还原
result_df.to_excel('微观性能预测结果.xlsx', index=False)#保存到表格

/var/folders/sy/gl301n8s5pl2fptdx8bcns5h0000gn/T/ipykernel_81963/3595924002.py:12: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  result_df.to_excel('微观性能预测结果.xlsx', index=False)#保存到表格


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors
import matplotlib.ticker
from pathlib import Path

# 读取数据
df = pd.read_excel('成分-强化相.xlsx')

# 提取Al, Ti, Nb和V列
a = 'Al'
b = 'Ti'
c = 'Nb'
target = 'V'

# 设置绘图风格
sns.set_context("notebook", font_scale=1.5, rc={"lines.linewidth": 2.5, "axes.labelsize": "large"})

# 设定常数
n = 1
cmap = 'YlGnBu'
norm = matplotlib.colors.BoundaryNorm(np.linspace(df[target].min(), df[target].max(), 4), 3)
fmt = matplotlib.ticker.FuncFormatter(lambda x, _: ['OTHERS', 'AC', 'QC'][norm(x)])

# 定义三元图绘制函数
def plot_tri(x, y, z, c, xlabel, ylabel, zlabel, cmap, cbarlabel=None, percentage=False, ax=None, **kwargs):
    # 绘制三元图的具体实现
    pass  # 这里填入你现有的plot_tri函数的代码

# 创建图形
fig, ax = plt.subplots(figsize=(10, 8), dpi=100)

# 绘制三元图
plot_tri(
    df[a].values * n,
    df[b].values * n,
    df[c].values * n,
    df[target].values,
    a, b, c,
    cbar_kw=dict(
        format=fmt,
        ticks=np.arange(df[target].min(), df[target].max(), step=(df[target].max() - df[target].min()) / 5)
    ),
    cmap=plt.get_cmap(cmap, 3),
    norm=norm,
    half_size=False,
    reduce_ticks=True,
    ax=ax
)

# 保存图形
Path('phase_diagram/').mkdir(exist_ok=True, parents=True)
plt.savefig(f"phase_diagram/Al_Ti_Nb_{target}.png", bbox_inches='tight', pad_inches=0, dpi=300)
plt.show()